# VizWiz Image Captioning — Phase 2: Model 1
## ResNet50 Encoder + GRU Decoder (Baseline)

**Author:** Shreyash  
**Assignment:** AT3 — Visual Question Answering & Image Captioning  
**Phase:** 2 (Model Training)  
**Date:** 2025

---

### Overview
This notebook implements **Model 1**, the baseline image captioning model:
- **Encoder:** ResNet50 (pretrained, frozen)
- **Decoder:** GRU with teacher forcing
- **Loss:** Cross-entropy (ignoring padding)
- **Evaluation:** BLEU-1 to BLEU-4 metrics

This is the foundational model before introducing attention and regularisation in Phase 3.

## 1. Imports and Reproducibility Setup

In [1]:
import os
import sys
import json
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

# PyTorch and torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Numerical and visualisation
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image

# Progress bars and evaluation
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)

print(f"PyTorch version: {torch.__version__}")
print("MPS available:", torch.backends.mps.is_available())

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

PyTorch version: 2.12.0
MPS available: True
Device: mps


## 2. Reproducibility and Device Setup

In [2]:
# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon (MPS)")
else:
    device = torch.device('cpu')
    print("Using CPU (training will be slower)")

print(f"Device: {device}")

Using Apple Silicon (MPS)
Device: mps


## 3. Project Path Configuration

In [3]:
# Determine project root (works from both project root and notebooks/ directory)
current_dir = Path.cwd()

# If we're in notebooks/, go up one level
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

# Define key directories
data_dir = project_root / 'data'
raw_data_dir = data_dir / 'raw'
processed_data_dir = data_dir / 'processed'
artefacts_dir = processed_data_dir / 'artefacts'
models_dir = project_root / 'models'
notebooks_dir = project_root / 'notebooks'

# Create models directory if it doesn't exist
models_dir.mkdir(exist_ok=True)

# Display paths
print("Project Structure:")
print(f"  Project Root   : {project_root}")
print(f"  Data Directory : {data_dir}")
print(f"  Artefacts      : {artefacts_dir}")
print(f"  Models         : {models_dir}")
print("")
print(f"Verifying key paths exist:")
print(f"  artefacts_dir exists : {artefacts_dir.exists()}")
print(f"  raw_data_dir exists  : {raw_data_dir.exists()}")

Project Structure:
  Project Root   : /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3
  Data Directory : /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3/data
  Artefacts      : /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3/data/processed/artefacts
  Models         : /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3/models

Verifying key paths exist:
  artefacts_dir exists : True
  raw_data_dir exists  : True


## 4. Load Phase 1 Artefacts

In [4]:
# Load vocabulary
vocab_path = artefacts_dir / 'vocab.pkl'
with open(vocab_path, 'rb') as f:
    vocab = pickle.load(f)

vocab_size = len(vocab)
print(f"Vocabulary loaded: {vocab_size} tokens")
print(f"Sample tokens: {list(vocab.items())[:5]}")

# Create word to index and index to word mappings
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# Special tokens
PAD_IDX = word2idx.get('<PAD>', 0)
START_IDX = word2idx.get('<START>', 1)
END_IDX = word2idx.get('<END>', 2)
UNK_IDX = word2idx.get('<UNK>', 3)

print(f"\nSpecial token indices:")
print(f"  PAD: {PAD_IDX}")
print(f"  START: {START_IDX}")
print(f"  END: {END_IDX}")
print(f"  UNK: {UNK_IDX}")

Vocabulary loaded: 7 tokens
Sample tokens: [('itos', {0: '<PAD>', 1: '<START>', 2: '<END>', 3: '<UNK>', 4: '0', 5: '00', 6: '05', 7: '07', 8: '1', 9: '10', 10: '100', 11: '11', 12: '12', 13: '14', 14: '15', 15: '150', 16: '16', 17: '17', 18: '18', 19: '2', 20: '20', 21: '200', 22: '2007', 23: '2011', 24: '2012', 25: '2013', 26: '22', 27: '24', 28: '25', 29: '27', 30: '28', 31: '2x', 32: '3', 33: '30', 34: '32', 35: '4', 36: '40', 37: '400', 38: '43', 39: '475', 40: '5', 41: '50', 42: '500', 43: '6', 44: '69', 45: '7', 46: '70', 47: '72', 48: '75', 49: '78', 50: '8', 51: '80', 52: '9', 53: '90', 54: '96', 55: 'a', 56: 'about', 57: 'above', 58: 'abstract', 59: 'ac', 60: 'accents', 61: 'access', 62: 'accessibility', 63: 'accessory', 64: 'account', 65: 'acer', 66: 'acetaminophen', 67: 'acid', 68: 'acoustic', 69: 'across', 70: 'action', 71: 'active', 72: 'ad', 73: 'adapter', 74: 'add', 75: 'added', 76: 'address', 77: 'ads', 78: 'adult', 79: 'advanced', 80: 'advertisement', 81: 'advertising'

In [6]:
# Load dataset splits
splits_path = artefacts_dir / 'splits.json'
with open(splits_path, 'r') as f:
    splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']
test_ids = splits['test']

print(f"Dataset splits:")
print(f"  Training IDs: {len(train_ids)}")
print(f"  Validation IDs: {len(val_ids)}")
print(f"  Test IDs: {len(test_ids)}")
print(f"  Total: {len(train_ids) + len(val_ids) + len(test_ids)}")

Dataset splits:
  Training IDs: 5425
  Validation IDs: 1162
  Test IDs: 1163
  Total: 7750


## 5. Dataset and DataLoader Setup

In [7]:
class VizWizCaptionDataset(Dataset):
    """
    VizWiz image captioning dataset.
    
    Loads images and returns their preprocessed tensors along with
    tokenised captions padded to max_length.
    """
    
    def __init__(self, image_dir, image_ids, captions_dict, word2idx, transform=None, max_length=50):
        """
        Args:
            image_dir (Path): Directory containing images
            image_ids (list): List of image IDs in this split
            captions_dict (dict): Mapping from image_id to list of captions
            word2idx (dict): Word to index mapping
            transform (callable): Image transformations
            max_length (int): Maximum caption length (padding/truncation)
        """
        self.image_dir = Path(image_dir)
        self.image_ids = image_ids
        self.captions_dict = captions_dict
        self.word2idx = word2idx
        self.transform = transform
        self.max_length = max_length
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.image_dir / f"{image_id}.jpg"
        
        # Load and transform image
        try:
            image = Image.open(image_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            # Return a placeholder
            image = torch.zeros(3, 224, 224)
        
        # Get first caption (you can implement multi-caption sampling here)
        captions = self.captions_dict.get(image_id, ['<START> <END>'])
        caption = captions[0] if captions else '<START> <END>'
        
        # Tokenise caption
        tokens = caption.split()
        token_ids = [self.word2idx.get(token, self.word2idx.get('<UNK>', 3)) for token in tokens]
        
        # Pad or truncate
        if len(token_ids) < self.max_length:
            token_ids = token_ids + [self.word2idx.get('<PAD>', 0)] * (self.max_length - len(token_ids))
        else:
            token_ids = token_ids[:self.max_length]
        
        caption_tensor = torch.tensor(token_ids, dtype=torch.long)
        
        return image, caption_tensor, image_id


def collate_fn(batch):
    """
    Custom collate function for DataLoader.
    Stacks images and captions into batches.
    """
    images, captions, image_ids = zip(*batch)
    images = torch.stack(images, dim=0)
    captions = torch.stack(captions, dim=0)
    return images, captions, image_ids


print("Dataset class defined.")

Dataset class defined.


## 6. Image Transformations

In [8]:
# Standard ImageNet normalization (for ResNet50)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transformations (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/test transformations (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Image transformations configured.")

Image transformations configured.


In [9]:
# Phase 1 has created captions_dict.pkl
captions_dict_path = artefacts_dir / 'captions_dict.pkl'

if captions_dict_path.exists():
    with open(captions_dict_path, 'rb') as f:
        captions_dict = pickle.load(f)
    print(f"Captions dictionary loaded: {len(captions_dict)} images with captions")
else:
    print(f"⚠ Captions dictionary not found at {captions_dict_path}")
    print("Creating minimal captions dictionary for demonstration...")
    # Create minimal captions for dataset compatibility
    captions_dict = {image_id: [f'<START> image {image_id} <END>'] 
                     for image_id in train_ids + val_ids + test_ids}
    print(f"Created minimal captions for {len(captions_dict)} images")

⚠ Captions dictionary not found at /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3/data/processed/artefacts/captions_dict.pkl
Creating minimal captions dictionary for demonstration...
Created minimal captions for 7750 images


In [10]:
captions_dict_path = artefacts_dir / "captions_dict.pkl"

if not captions_dict_path.exists():
    raise FileNotFoundError(
        f"captions_dict.pkl not found at {captions_dict_path}. "
        "Create it in Phase 1 or load captions directly from data/raw/annotations/val.json."
    )

with open(captions_dict_path, "rb") as f:
    captions_dict = pickle.load(f)

print(f"Captions dictionary loaded: {len(captions_dict)} images with captions")

FileNotFoundError: captions_dict.pkl not found at /Users/shreyash/Documents/Sem3DL/AT3DL/vizwiz-val-image-cap-DL-AT3/data/processed/artefacts/captions_dict.pkl. Create it in Phase 1 or load captions directly from data/raw/annotations/val.json.

In [ ]:
# Create datasets
train_dataset = VizWizCaptionDataset(
    image_dir=raw_data_dir / 'train',
    image_ids=train_ids,
    captions_dict=captions_dict,
    word2idx=word2idx,
    transform=train_transform,
    max_length=50
)

val_dataset = VizWizCaptionDataset(
    image_dir=raw_data_dir / 'val',
    image_ids=val_ids,
    captions_dict=captions_dict,
    word2idx=word2idx,
    transform=val_transform,
    max_length=50
)

test_dataset = VizWizCaptionDataset(
    image_dir=raw_data_dir / 'test',
    image_ids=test_ids,
    captions_dict=captions_dict,
    word2idx=word2idx,
    transform=val_transform,
    max_length=50
)

print(f"Datasets created:")
print(f"  Train: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")
print(f"  Test: {len(test_dataset)}")

## 7. Phase 2 — Model 1: ResNet50 Encoder + GRU Decoder

### Architecture Overview

**Encoder:** ResNet50
- Pretrained on ImageNet
- Output dimension: 2048 (global average pooled)
- **Frozen** (not trainable, transfer learning)

**Decoder:** GRU
- Input: word embeddings + attention context (here just encoder output)
- Hidden units: 512
- Single layer
- Output: vocabulary logits

**Training:** Teacher forcing
- Ground truth tokens fed at each timestep
- Enables faster convergence

**Inference:** Greedy decoding
- Argmax selection at each step
- Simple baseline (no beam search)

## 8. Model 1 Architecture

In [ ]:
class ResNet50Encoder(nn.Module):
    """
    ResNet50 image encoder.
    
    Returns a global average pooled feature vector of dimension 2048.
    Weights are pretrained on ImageNet and frozen for transfer learning.
    """
    
    def __init__(self, freeze=True):
        super(ResNet50Encoder, self).__init__()
        
        # Load pretrained ResNet50
        resnet = models.resnet50(pretrained=True)
        
        # Remove the final classification layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        # Freeze all parameters if freeze=True
        if freeze:
            for param in self.features.parameters():
                param.requires_grad = False
        
        self.output_dim = 2048
    
    def forward(self, images):
        """
        Args:
            images: Tensor of shape (batch_size, 3, 224, 224)
        
        Returns:
            features: Tensor of shape (batch_size, 2048)
        """
        # Extract features: (batch, 2048, 1, 1)
        x = self.features(images)
        # Flatten: (batch, 2048)
        x = x.view(x.size(0), -1)
        return x


class GRUDecoder(nn.Module):
    """
    GRU-based caption decoder.
    
    Takes image features and generates captions token-by-token using
    Gated Recurrent Units.
    """
    
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, encoder_dim=2048):
        super(GRUDecoder, self).__init__()
        
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.encoder_dim = encoder_dim
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Project encoder output to hidden state
        self.encoder_to_hidden = nn.Linear(encoder_dim, hidden_dim)
        
        # GRU layer
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
        # Output projection to vocabulary
        self.output_projection = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, encoder_features, captions):
        """
        Forward pass with teacher forcing (training).
        
        Args:
            encoder_features: Tensor of shape (batch_size, encoder_dim)
            captions: Tensor of shape (batch_size, max_length)
        
        Returns:
            outputs: Tensor of shape (batch_size, max_length, vocab_size)
        """
        batch_size = encoder_features.size(0)
        max_length = captions.size(1)
        
        # Initialise hidden state from encoder features
        hidden = self.encoder_to_hidden(encoder_features).unsqueeze(0)  # (1, batch, hidden_dim)
        
        # Embed captions
        embeddings = self.embedding(captions)  # (batch, max_length, embed_dim)
        
        # Forward through GRU
        gru_output, _ = self.gru(embeddings, hidden)  # (batch, max_length, hidden_dim)
        
        # Project to vocabulary
        outputs = self.output_projection(gru_output)  # (batch, max_length, vocab_size)
        
        return outputs
    
    def generate_caption(self, encoder_features, max_length=50, temperature=1.0):
        """
        Greedy caption generation (inference).
        
        Args:
            encoder_features: Tensor of shape (1, encoder_dim)
            max_length: Maximum caption length
            temperature: Softmax temperature (unused for greedy, for future extension)
        
        Returns:
            caption_ids: List of token IDs
        """
        caption_ids = [START_IDX]
        hidden = self.encoder_to_hidden(encoder_features).unsqueeze(0)
        
        for _ in range(max_length - 1):
            # Embed last token
            last_token = torch.tensor([caption_ids[-1]], dtype=torch.long, device=encoder_features.device)
            embedding = self.embedding(last_token).unsqueeze(0)  # (1, 1, embed_dim)
            
            # GRU forward
            gru_output, hidden = self.gru(embedding, hidden)
            
            # Predict next token
            logits = self.output_projection(gru_output)  # (1, 1, vocab_size)
            next_token_id = logits.argmax(dim=-1).item()
            
            caption_ids.append(next_token_id)
            
            # Stop if END token is generated
            if next_token_id == END_IDX:
                break
        
        return caption_ids


class EncoderDecoder(nn.Module):
    """
    Complete encoder-decoder model for image captioning.
    """
    
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super(EncoderDecoder, self).__init__()
        self.encoder = ResNet50Encoder(freeze=True)
        self.decoder = GRUDecoder(vocab_size, embed_dim, hidden_dim, self.encoder.output_dim)
    
    def forward(self, images, captions):
        """Forward pass during training."""
        encoder_features = self.encoder(images)
        outputs = self.decoder(encoder_features, captions)
        return outputs
    
    def generate(self, images, max_length=50):
        """Generate captions for images (inference)."""
        encoder_features = self.encoder(images)  # (batch, 2048)
        captions = []
        for i in range(encoder_features.size(0)):
            caption = self.decoder.generate_caption(encoder_features[i:i+1], max_length)
            captions.append(caption)
        return captions


print("Model classes defined.")

## 9. Training Configuration

In [ ]:
# Hyperparameters
CONFIG = {
    'batch_size': 16,  # Conservative for Mac/smaller GPUs
    'num_epochs': 10,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'embed_dim': 256,
    'hidden_dim': 512,
    'max_caption_length': 50,
    'num_workers': 0,  # Set to 0 for compatibility (Windows/Mac with MPS)
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn
)

print(f"DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Instantiate model
model = EncoderDecoder(
    vocab_size=vocab_size,
    embed_dim=CONFIG['embed_dim'],
    hidden_dim=CONFIG['hidden_dim']
)

# Move to device
model = model.to(device)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = count_parameters(model)
total_params = sum(p.numel() for p in model.parameters())

print(f"Model initialised on {device}:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters: {total_params - trainable_params:,}")
print(f"\nModel structure:")
print(f"  Encoder: ResNet50 (frozen)")
print(f"  Decoder: GRU (trainable)")

In [ ]:
# Loss function (ignore PAD tokens)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, reduction='mean')

# Optimizer (only train decoder)
optimizer = torch.optim.Adam(
    model.decoder.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler (optional)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True
)

print("Training setup complete:")
print(f"  Loss: CrossEntropyLoss (ignoring PAD={PAD_IDX})")
print(f"  Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"  Scheduler: ReduceLROnPlateau")

## 10. Training and Validation Loops

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Single training epoch.
    """
    model.train()
    total_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc="Training", leave=True)
    
    for batch_idx, (images, captions, image_ids) in enumerate(progress_bar):
        # Move to device
        images = images.to(device)
        captions = captions.to(device)
        
        # Forward pass
        outputs = model(images, captions)
        # outputs shape: (batch, max_length, vocab_size)
        # captions shape: (batch, max_length)
        
        # Reshape for loss calculation
        batch_size, max_length, vocab_size = outputs.shape
        outputs_reshaped = outputs.view(-1, vocab_size)
        captions_reshaped = captions.view(-1)
        
        # Compute loss
        loss = criterion(outputs_reshaped, captions_reshaped)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item():.4f})
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss


def validate(model, val_loader, criterion, device):
    """
    Validation loop.
    """
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc="Validation", leave=True)
        
        for images, captions, image_ids in progress_bar:
            # Move to device
            images = images.to(device)
            captions = captions.to(device)
            
            # Forward pass
            outputs = model(images, captions)
            
            # Reshape and compute loss
            batch_size, max_length, vocab_size = outputs.shape
            outputs_reshaped = outputs.view(-1, vocab_size)
            captions_reshaped = captions.view(-1)
            
            loss = criterion(outputs_reshaped, captions_reshaped)
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item():.4f})
    
    avg_loss = total_loss / len(val_loader)
    return avg_loss


print("Training loop functions defined.")

## 11. Train Model 1

Training the baseline ResNet50 + GRU model with teacher forcing.

In [ ]:
# Track metrics
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_epoch = 0

print(f"\n{'='*60}")
print(f"Training Model 1: ResNet50 + GRU (Baseline)")
print(f"{'='*60}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Device: {device}")
print(f"{'='*60}\n")

# Training loop
for epoch in range(1, CONFIG['num_epochs'] + 1):
    print(f"\n[Epoch {epoch}/{CONFIG['num_epochs']}]")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    print(f"Training Loss: {train_loss:.4f}")
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    print(f"Validation Loss: {val_loss:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        checkpoint_path = models_dir / 'shreyash_model1_resnet50_gru.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, checkpoint_path)
        print(f"✓ Best checkpoint saved (Val Loss: {val_loss:.4f})")

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best epoch: {best_epoch} (Val Loss: {best_val_loss:.4f})")
print(f"{'='*60}")

## 12. Plot Loss Curves

In [ ]:
# Plot training and validation loss
fig, ax = plt.subplots(figsize=(10, 5))

epochs_range = range(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=5)
ax.plot(epochs_range, val_losses, 'r-s', label='Validation Loss', linewidth=2, markersize=5)

# Mark best epoch
ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (CrossEntropy)', fontsize=12)
ax.set_title('Model 1: Training and Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(models_dir / 'model1_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Loss curves saved to {models_dir / 'model1_loss_curves.png'}")

## 13. Caption Generation Function

In [ ]:
def decode_caption(caption_ids, idx2word, end_idx=END_IDX):
    """
    Convert a list of token IDs to a readable caption string.
    
    Args:
        caption_ids: List of token IDs
        idx2word: Index to word mapping
        end_idx: Token ID for <END>
    
    Returns:
        caption: String caption
    """
    words = []
    for token_id in caption_ids:
        if token_id == end_idx:
            break
        if token_id in idx2word:
            word = idx2word[token_id]
            if word not in ['<START>', '<PAD>', '<UNK>']:
                words.append(word)
    return ' '.join(words)


print("Caption decoding function defined.")

## 14. BLEU-1 to BLEU-4 Evaluation

In [ ]:
# Ensure NLTK data is available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("NLTK resources ready.")

In [ ]:
def compute_bleu_scores(generated_captions, reference_captions, max_n=4):
    """
    Compute BLEU-1 to BLEU-n scores.
    
    Args:
        generated_captions: List of generated caption strings
        reference_captions: List of reference caption strings
        max_n: Maximum n-gram (default 4 for BLEU-4)
    
    Returns:
        bleu_scores: Dictionary with BLEU-1, BLEU-2, ..., BLEU-n
    """
    bleu_scores = {f'BLEU-{i}': [] for i in range(1, max_n + 1)}
    smoothing = SmoothingFunction().method1
    
    for gen_cap, ref_cap in zip(generated_captions, reference_captions):
        # Tokenise
        gen_tokens = gen_cap.lower().split()
        ref_tokens = ref_cap.lower().split()
        
        # Compute BLEU scores for each n
        for n in range(1, max_n + 1):
            weights = tuple([1.0 / n] * n)
            bleu_score = sentence_bleu(
                [ref_tokens],
                gen_tokens,
                weights=weights,
                smoothing_function=smoothing
            )
            bleu_scores[f'BLEU-{n}'].append(bleu_score)
    
    # Average across all samples
    final_scores = {key: np.mean(values) for key, values in bleu_scores.items()}
    return final_scores


print("BLEU evaluation function defined.")

In [ ]:
# Load best checkpoint
checkpoint_path = models_dir / 'shreyash_model1_resnet50_gru.pth'
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best checkpoint from {checkpoint_path}")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Validation Loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Generate captions on validation set and compute BLEU scores
model.eval()
generated_captions_list = []
reference_captions_list = []

print("Generating captions on validation set...")

with torch.no_grad():
    for images, captions, image_ids in tqdm(val_loader, desc="Generating", leave=True):
        images = images.to(device)
        
        # Generate captions
        generated_caption_ids = model.generate(images, max_length=50)
        
        for i, (gen_ids, img_id) in enumerate(zip(generated_caption_ids, image_ids)):
            # Decode generated caption
            gen_caption = decode_caption(gen_ids, idx2word)
            generated_captions_list.append(gen_caption)
            
            # Get reference caption
            ref_caption = captions_dict.get(img_id, ['<START> <END>'])[0]
            ref_caption = decode_caption(ref_caption.split(), idx2word)
            reference_captions_list.append(ref_caption)

print(f"Generated {len(generated_captions_list)} captions.")

In [ ]:
# Compute BLEU scores
print("\nComputing BLEU scores...")
bleu_scores = compute_bleu_scores(generated_captions_list, reference_captions_list, max_n=4)

print("\n" + "="*50)
print("Model 1: BLEU Evaluation Results")
print("="*50)
for metric, score in bleu_scores.items():
    print(f"{metric}: {score:.4f}")
print("="*50)

## 15. Qualitative Results

In [ ]:
# Display some examples
num_examples = 6
fig, axes = plt.subplots(num_examples, 1, figsize=(14, 4 * num_examples))

if num_examples == 1:
    axes = [axes]

for idx in range(min(num_examples, len(generated_captions_list))):
    ax = axes[idx]
    
    gen_caption = generated_captions_list[idx]
    ref_caption = reference_captions_list[idx]
    
    # Display caption information
    ax.text(0.5, 0.6, f"Generated: {gen_caption}", 
            ha='center', va='center', fontsize=11, wrap=True,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    ax.text(0.5, 0.3, f"Reference: {ref_caption}", 
            ha='center', va='center', fontsize=11, wrap=True,
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(f"Example {idx + 1}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(models_dir / 'model1_qualitative_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Qualitative results saved to {models_dir / 'model1_qualitative_results.png'}")

## 16. Save Model 1 Checkpoint and Metadata

In [ ]:
# Save comprehensive checkpoint with metadata
checkpoint_path = models_dir / 'shreyash_model1_resnet50_gru.pth'

checkpoint_data = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': CONFIG,
    'vocab_size': vocab_size,
    'train_loss': train_losses[best_epoch - 1] if best_epoch <= len(train_losses) else None,
    'val_loss': best_val_loss,
    'bleu_scores': bleu_scores,
}

torch.save(checkpoint_data, checkpoint_path)

print(f"\n✓ Model 1 checkpoint saved to:")
print(f"  {checkpoint_path}")
print(f"\nCheckpoint contains:")
print(f"  - Model state dict")
print(f"  - Optimizer state dict")
print(f"  - Configuration")
print(f"  - BLEU scores")
print(f"  - Training metadata")

In [ ]:
# Save metadata to JSON
metadata = {
    'model_name': 'Model 1: ResNet50 + GRU (Baseline)',
    'phase': 2,
    'architecture': {
        'encoder': 'ResNet50 (pretrained, frozen)',
        'decoder': 'GRU',
        'encoder_output_dim': 2048,
        'embedding_dim': CONFIG['embed_dim'],
        'hidden_dim': CONFIG['hidden_dim'],
        'attention': False,
        'dropout': False,
    },
    'training': {
        'epochs': CONFIG['num_epochs'],
        'batch_size': CONFIG['batch_size'],
        'learning_rate': CONFIG['learning_rate'],
        'optimizer': 'Adam',
        'loss': 'CrossEntropyLoss (ignore PAD)',
        'scheduler': 'ReduceLROnPlateau',
        'decoding': 'Greedy (argmax)',
        'teacher_forcing': True,
    },
    'results': {
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'bleu_scores': bleu_scores,
    },
    'dataset': {
        'train_size': len(train_ids),
        'val_size': len(val_ids),
        'test_size': len(test_ids),
        'vocab_size': vocab_size,
        'max_caption_length': CONFIG['max_caption_length'],
    }
}

metadata_path = models_dir / 'model1_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to {metadata_path}")

## 17. Model 1 Discussion and Limitations

### Summary

**Model 1** is the baseline image captioning model combining:
- **ResNet50 Encoder:** Transfer learning with frozen pretrained weights
- **GRU Decoder:** Simple recurrent architecture with teacher forcing

### Key Results

| Metric | Value |
|--------|-------|
| Best Epoch | {0} |
| Validation Loss | {1:.4f} |
| BLEU-1 | {2:.4f} |
| BLEU-2 | {3:.4f} |
| BLEU-3 | {4:.4f} |
| BLEU-4 | {5:.4f} |

### Strengths
1. **Stable training:** Teacher forcing ensures convergence
2. **Efficient:** Frozen encoder reduces computational load
3. **Clean baseline:** No architectural complexity for future comparisons

### Limitations
1. **No attention mechanism:** Cannot focus on relevant image regions
2. **No regularisation:** Prone to overfitting on small datasets
3. **Greedy decoding:** Suboptimal at inference time (no beam search)
4. **Fixed encoder:** May limit expressiveness for specialized domains
5. **Single-step generation:** Cannot leverage past generated context fully

### Next Steps (Phase 3)
Model 2 will introduce:
- **Attention mechanism** for spatial reasoning
- **Dropout regularisation** to reduce overfitting
- **Fine-tuning** of encoder layers
- Potential beam search decoding

---

In [ ]:
# Print summary
print("\n" + "="*70)
print("MODEL 1 TRAINING SUMMARY")
print("="*70)
print(f"\nArchitecture: ResNet50 (frozen) + GRU")
print(f"\nTraining Results:")
print(f"  Best Epoch: {best_epoch}")
print(f"  Best Validation Loss: {best_val_loss:.4f}")
print(f"\nBLEU Scores (Validation Set):")
for metric, score in bleu_scores.items():
    print(f"  {metric}: {score:.4f}")
print(f"\nCheckpoint: {checkpoint_path}")
print(f"Loss curves: {models_dir / 'model1_loss_curves.png'}")
print(f"Qualitative results: {models_dir / 'model1_qualitative_results.png'}")
print(f"Metadata: {metadata_path}")
print("\n" + "="*70)

## 18. Phase 3 Placeholder — Model 2: ResNet50 + Attention GRU + Dropout

### Coming Soon

Model 2 will extend Model 1 with the following improvements:

#### Architecture Enhancements
1. **Attention Mechanism**
   - Compute attention weights over encoder features
   - Enable decoder to focus on relevant image regions
   - Bahdanau-style attention (additive)

2. **Dropout Regularisation**
   - Embedding dropout (p=0.3)
   - GRU dropout (p=0.3)
   - Output dropout (p=0.2)
   - Reduces overfitting

3. **Fine-tuning**
   - Unfreeze ResNet50 layer4
   - Lower learning rate for encoder (1e-5)
   - Higher learning rate for decoder (1e-3)

#### Expected Improvements
- Higher BLEU scores (especially BLEU-3 and BLEU-4)
- Better generalisation (lower overfitting)
- More interpretable (attention visualisation)
- Improved caption quality

#### Training Notes
- Longer training required (15+ epochs)
- Monitor for overfitting
- Potential for beam search decoding

---

**Status:** ⏳ Phase 3 implementation pending  
**Priority:** High — Expected to improve BLEU scores by 5-10%  
**Estimated completion:** Next phase of assignment

In [ ]:
# Placeholder for Phase 3 Model 2 implementation
print("\n" + "="*70)
print("PHASE 3 PLACEHOLDER")
print("="*70)
print("\nModel 2: ResNet50 + Attention GRU + Dropout")
print("\nImplementation checklist:")
print("  [ ] Attention mechanism (Bahdanau)")
print("  [ ] Dropout layers")
print("  [ ] Fine-tuning strategy")
print("  [ ] Training loop adjustments")
print("  [ ] Evaluation and comparison")
print("  [ ] Attention visualisation")
print("\nExpected improvements over Model 1:")
print("  - +5-10% BLEU score improvement")
print("  - Better generalisation")
print("  - Interpretable attention weights")
print("\n" + "="*70)